# ADLINK PCI-9812 — DASK API Radar Sampler
Translated directly from the working C reference code.  
Uses `DASK.dll` (installed with the ADLINK DASK driver package).

**Flow:** `Register_Card` → `AI_9812_Config` → `AI_AsyncDblBufferMode` → `AI_ContScanChannelsToFile` → poll `AI_AsyncDblBufferHalfReady` → `AI_AsyncClear` → `Release_Card`

In [ ]:
import sys, ctypes, time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

if sys.platform != 'win32':
    raise EnvironmentError('This notebook must run on Windows (DASK.dll required)')

print('Python', sys.version)

## Constants — from `dask.h`
These match the defines in `dask.h` shipped with the DASK driver.  
Verify against your installed header if you see unexpected behaviour.

In [ ]:
# ---- Card type ----
PCI_9812 = 17

# ---- A/D input range ----
AD_B_5_V  = 1   # Bipolar ±5 V
AD_B_1_V  = 3   # Bipolar ±1 V

# ---- Trigger mode (P9812_TRGMOD_*) ----
P9812_TRGMOD_SOFT = 0   # Software trigger
P9812_TRGMOD_POST = 1   # Post-trigger
P9812_TRGMOD_PRE  = 2   # Pre-trigger
P9812_TRGMOD_MIDL = 3   # Middle-trigger
P9812_TRGMOD_DELY = 4   # Delay-trigger

# ---- Trigger source (P9812_TRGSRC_*) ----
P9812_TRGSRC_CH0  = 0
P9812_TRGSRC_CH1  = 1
P9812_TRGSRC_CH2  = 2
P9812_TRGSRC_CH3  = 3
P9812_TRGSRC_EXT  = 4   # External digital trigger

# ---- Trigger slope (P9812_TRGSLP_*) ----
P9812_TRGSLP_POS  = 0   # Rising edge
P9812_TRGSLP_NEG  = 1   # Falling edge

# ---- AD timing / clock (P9812_AD2_* | P9812_CLKSRC_*) ----
P9812_CLKSRC_INT  = 0x0000   # Internal clock
P9812_CLKSRC_SIN  = 0x0004   # External sine wave
P9812_CLKSRC_SQR  = 0x0008   # External square wave
P9812_AD2_GT_PCI  = 0x0002   # AD clock gated by PCI bus

# ---- Operation mode ----
SYNCH_OP   = 0
ASYNCH_OP  = 1

print('Constants loaded.')

## DASK DLL Wrapper

In [ ]:
class DASK:
    """
    Thin ctypes wrapper around DASK.dll.
    Mirrors the C API used in the reference program exactly.
    """

    DLL_NAME = 'DASK.dll'

    def __init__(self):
        try:
            self._dll = ctypes.windll.LoadLibrary(self.DLL_NAME)
        except OSError as exc:
            raise RuntimeError(
                f'Cannot load {self.DLL_NAME}.\n'
                'Install the ADLINK DASK driver package and try again.'
            ) from exc
        self._set_prototypes()
        print(f'{self.DLL_NAME} loaded.')

    # ------------------------------------------------------------------ API

    def Register_Card(self, card_type, card_num):
        """
        Returns card handle (I16). Negative value = error.
        Equivalent to:  card = Register_Card(PCI_9812, card_num)
        """
        handle = self._dll.Register_Card(
            ctypes.c_uint16(card_type),
            ctypes.c_uint16(card_num),
        )
        if handle < 0:
            raise RuntimeError(f'Register_Card failed, error={handle}')
        print(f'Card registered, handle={handle}')
        return handle

    def AI_9812_Config(self, card, trig_mod, trig_src, trig_slp,
                       ad_timing, trig_level, post_count):
        """
        Configure PCI-9812 analog input.
        ad_timing = P9812_AD2_GT_PCI | P9812_CLKSRC_INT  (typical)
        trig_level = 0x80  (midpoint for analog trigger)
        """
        err = self._dll.AI_9812_Config(
            ctypes.c_int16(card),
            ctypes.c_uint16(trig_mod),
            ctypes.c_uint16(trig_src),
            ctypes.c_uint16(trig_slp),
            ctypes.c_uint16(ad_timing),
            ctypes.c_uint16(trig_level),
            ctypes.c_uint32(post_count),
        )
        self._check(err, 'AI_9812_Config')

    def AI_AsyncDblBufferMode(self, card, enable):
        """Enable (1) or disable (0) double-buffer acquisition mode."""
        err = self._dll.AI_AsyncDblBufferMode(
            ctypes.c_int16(card),
            ctypes.c_uint16(1 if enable else 0),
        )
        self._check(err, 'AI_AsyncDblBufferMode')

    def AI_ContScanChannelsToFile(self, card, channel, ad_range,
                                   file_name, count, sample_rate, synch):
        """
        Start continuous multi-channel scan, storing to file.
        channel    = last channel index (0 → 4 channels: CH0–CH3)
        count      = circular buffer size (samples, all channels combined)
        sample_rate= Hz per channel
        file_name  = base name (no extension); driver appends '.dat'
        """
        if isinstance(file_name, str):
            file_name = file_name.encode('ascii')
        err = self._dll.AI_ContScanChannelsToFile(
            ctypes.c_int16(card),
            ctypes.c_uint16(channel),
            ctypes.c_uint16(ad_range),
            ctypes.c_char_p(file_name),
            ctypes.c_uint32(count),
            ctypes.c_double(sample_rate),
            ctypes.c_uint16(synch),
        )
        self._check(err, 'AI_ContScanChannelsToFile')

    def AI_AsyncDblBufferHalfReady(self, card):
        """
        Returns (half_ready: bool, fStop: bool).
        half_ready = True when half the circular buffer is filled.
        fStop      = True when the card has stopped on its own.
        """
        half_ready = ctypes.c_uint16(0)
        f_stop     = ctypes.c_uint16(0)
        self._dll.AI_AsyncDblBufferHalfReady(
            ctypes.c_int16(card),
            ctypes.byref(half_ready),
            ctypes.byref(f_stop),
        )
        return bool(half_ready.value), bool(f_stop.value)

    def AI_AsyncDblBufferTransfer(self, card, buf=None):
        """
        Transfer the ready half-buffer.
        buf=None  → transfer to file (matches C code: AI_AsyncDblBufferTransfer(card, NULL))
        buf=array → transfer into a ctypes/numpy buffer
        """
        ptr = ctypes.cast(buf, ctypes.c_void_p) if buf is not None else None
        self._dll.AI_AsyncDblBufferTransfer(
            ctypes.c_int16(card),
            ptr,
        )

    def AI_AsyncClear(self, card):
        """Stop async acquisition. Returns total sample count."""
        count = ctypes.c_uint32(0)
        err   = self._dll.AI_AsyncClear(
            ctypes.c_int16(card),
            ctypes.byref(count),
        )
        self._check(err, 'AI_AsyncClear')
        return count.value

    def Release_Card(self, card):
        """Release the card handle."""
        err = self._dll.Release_Card(ctypes.c_int16(card))
        self._check(err, 'Release_Card')
        print(f'Card {card} released.')

    # ------------------------------------------------------------------ helpers

    def _set_prototypes(self):
        d   = self._dll
        I16 = ctypes.c_int16
        U16 = ctypes.c_uint16
        U32 = ctypes.c_uint32
        F64 = ctypes.c_double

        d.Register_Card.argtypes                = [U16, U16]
        d.Register_Card.restype                 = I16

        d.AI_9812_Config.argtypes               = [I16, U16, U16, U16, U16, U16, U32]
        d.AI_9812_Config.restype                = I16

        d.AI_AsyncDblBufferMode.argtypes        = [I16, U16]
        d.AI_AsyncDblBufferMode.restype         = I16

        d.AI_ContScanChannelsToFile.argtypes    = [I16, U16, U16, ctypes.c_char_p, U32, F64, U16]
        d.AI_ContScanChannelsToFile.restype     = I16

        d.AI_AsyncDblBufferHalfReady.argtypes   = [I16, ctypes.POINTER(U16), ctypes.POINTER(U16)]
        d.AI_AsyncDblBufferHalfReady.restype    = I16

        d.AI_AsyncDblBufferTransfer.argtypes    = [I16, ctypes.c_void_p]
        d.AI_AsyncDblBufferTransfer.restype     = I16

        d.AI_AsyncClear.argtypes                = [I16, ctypes.POINTER(U32)]
        d.AI_AsyncClear.restype                 = I16

        d.Release_Card.argtypes                 = [I16]
        d.Release_Card.restype                  = I16

    @staticmethod
    def _check(err, fname):
        if err != 0:
            raise RuntimeError(f'{fname} returned error code {err}')


print('DASK wrapper ready.')

## Parameters
Edit here — matches the variables at the top of the C source.

In [ ]:
channel     = 3             # Last channel index — 0 to 3 → scans CH0 … CHn
ad_range    = AD_B_5_V      # AD_B_5_V (±5V) or AD_B_1_V (±1V)
file_name   = '9812d'       # Output base name; driver writes '9812d.dat'
read_count  = 4000          # Circular buffer size (samples across all channels)
sample_rate = 20000.0       # Samples/second per channel
card_num    = int(input('Card number [0]: ') or 0)

n_channels  = channel + 1   # Number of channels being scanned
VRANGE      = 5.0 if ad_range == AD_B_5_V else 1.0
ADC_MID     = 2048          # 12-bit midpoint

print(f'Scanning CH0–CH{channel}  |  {sample_rate:.0f} S/s  |  '
      f'buffer={read_count}  |  output → {file_name}.dat')

## Acquire — Double-Buffer to File
Mirrors the C program exactly. Runs until `duration_s` seconds elapse  
(replaces `kbhit()` which is not available in Jupyter).

In [ ]:
duration_s = 5.0   # How long to acquire (seconds) — adjust as needed

dask = DASK()
card = dask.Register_Card(PCI_9812, card_num)

dask.AI_9812_Config(
    card,
    trig_mod   = P9812_TRGMOD_SOFT,
    trig_src   = P9812_TRGSRC_CH0,
    trig_slp   = P9812_TRGSLP_POS,
    ad_timing  = P9812_AD2_GT_PCI | P9812_CLKSRC_INT,
    trig_level = 0x80,
    post_count = 0,
)

dask.AI_AsyncDblBufferMode(card, enable=True)

dask.AI_ContScanChannelsToFile(
    card,
    channel     = channel,
    ad_range    = ad_range,
    file_name   = file_name,
    count       = read_count,
    sample_rate = sample_rate,
    synch       = ASYNCH_OP,
)

count       = 0
half_count  = read_count // 2
deadline    = time.monotonic() + duration_s

print(f'Acquiring for {duration_s}s …')
while time.monotonic() < deadline:
    half_ready, f_stop = dask.AI_AsyncDblBufferHalfReady(card)
    if half_ready:
        dask.AI_AsyncDblBufferTransfer(card, buf=None)  # transfer to file
        count += half_count
        print(f'  {count} samples transferred', end='\r')
    if f_stop:
        print('\nCard stopped early.')
        break

total = dask.AI_AsyncClear(card)
dask.Release_Card(card)

print(f'\nDone. {count} samples written to {file_name}.dat  (AI_AsyncClear count={total})')

## Read Back & Plot the `.dat` File
The DASK driver writes raw 16-bit ADC counts, interleaved across channels.

In [ ]:
raw = np.fromfile(f'{file_name}.dat', dtype=np.int16)
print(f'Read {len(raw)} samples from {file_name}.dat')

# De-interleave: CH0, CH1, CH2, CH3, CH0, CH1 …
ch_data = {}
for ch in range(n_channels):
    raw_ch = raw[ch::n_channels].astype(np.float32)
    ch_data[ch] = (raw_ch - ADC_MID) / ADC_MID * VRANGE   # → volts

# Time axis
t = np.arange(len(ch_data[0])) / sample_rate * 1e3   # ms

# Plot
fig, axes = plt.subplots(n_channels, 1, figsize=(14, 3 * n_channels), sharex=True)
if n_channels == 1:
    axes = [axes]

for ch, ax in enumerate(axes):
    ax.plot(t, ch_data[ch], linewidth=0.6)
    ax.set_ylabel(f'CH{ch} (V)')
    ax.set_ylim(-VRANGE * 1.05, VRANGE * 1.05)
    ax.axhline(0, color='gray', linewidth=0.4, linestyle='--')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (ms)')
fig.suptitle(f'PCI-9812  |  {sample_rate:.0f} S/s  |  {file_name}.dat', fontsize=13)
plt.tight_layout()
plt.show()

# Per-channel stats
print('\nChannel statistics:')
for ch, v in ch_data.items():
    print(f'  CH{ch}: min={v.min():.4f} V  max={v.max():.4f} V  '
          f'mean={v.mean():.4f} V  rms={np.sqrt(np.mean(v**2)):.4f} V')

## FFT Spectrum

In [ ]:
fig, axes = plt.subplots(n_channels, 1, figsize=(14, 3 * n_channels), sharex=True)
if n_channels == 1:
    axes = [axes]

for ch, ax in enumerate(axes):
    v   = ch_data[ch]
    N   = len(v)
    win = np.hanning(N)
    sp  = np.abs(np.fft.rfft(v * win)) * 2 / N
    f   = np.fft.rfftfreq(N, d=1.0 / sample_rate) / 1e3   # kHz
    ax.plot(f, 20 * np.log10(sp + 1e-9), linewidth=0.6)
    ax.set_ylabel(f'CH{ch} (dBV)')
    ax.set_ylim(-100, 10)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Frequency (kHz)')
fig.suptitle('Frequency Spectrum  |  Hanning window', fontsize=13)
plt.tight_layout()
plt.show()